# Lab 10 — Fine-tuning YOLO: from detections to millimetres

**Deep Learning for Computer Vision — Day 3** · ECAM, 5MAU · 2026–2027

This lab closes the loop of the course: **image → detector → (u, v) → calibration → (X, Y) in mm → robot**.

**Goals.**
1. Understand IoU and non-maximum suppression (NMS) — compute them yourself.
2. See why a COCO-pretrained detector and the hand-made detector of Day 1 are not enough.
3. **Fine-tune YOLO** on images of our work table (3 classes: `disc`, `plate`, `washer`).
4. Read the metrics: precision, recall, mAP@0.5, mAP@0.5:0.95.
5. Convert the detections to **millimetres** with the calibration of Day 1 and measure the error.
6. Train the **segmentation** variant and get the orientation of the plates for the gripper.

**Where to run it.** Training is ~10× faster on a GPU: open this notebook in **Google Colab** (*Runtime → Change runtime type → T4 GPU*) and upload `table_yolo.zip` when asked. It also runs on a laptop CPU with fewer epochs (≈ 10 min per training).

**Data.** `table_yolo`: 160 training / 40 validation images of the table rendered with the simulated USB camera of Day 1 (same intrinsics, same distortion, same ArUco markers), generated by `tools/make_yolo_dataset.py`. Ground-truth positions in mm are known for the validation images. The **washers** are grey metal rings: invisible to the HSV detector of Day 1.

> *References:* Ultralytics YOLO documentation, https://docs.ultralytics.com/ · J. Redmon *et al.*, "You Only Look Once: unified, real-time object detection", CVPR 2016 · Stanford CS231n, *Detection and segmentation* lecture · M. Everingham *et al.*, "The PASCAL Visual Object Classes challenge", IJCV 2010 (mAP) · T.-Y. Lin *et al.*, "Microsoft COCO", ECCV 2014.

In [ ]:
import sys

if "google.colab" in sys.modules:  # Colab: the packages are installed by the next cell
    print("Colab: setup check skipped.")
else:
    # Setup check: are all the packages of requirements.txt installed in this kernel?
    import importlib
    import importlib.util

    importlib.invalidate_caches()
    REQUIRED = {"torch": "torch", "torchvision": "torchvision", "ultralytics": "ultralytics", "numpy": "numpy",
                "matplotlib": "matplotlib", "cv2": "opencv-contrib-python"}
    missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
    assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                         "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
    print("Setup OK: all required packages are installed.")

In [ ]:
import json
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install ultralytics "opencv-contrib-python-headless<5"
    from google.colab import files
    if not Path("data/table_yolo").exists():
        print("Upload table_yolo.zip")
        files.upload()
        !mkdir -p data && unzip -q -o table_yolo.zip -d data

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.ops import box_iou, nms
from ultralytics import YOLO

DATA = Path("data") if Path("data").exists() else Path("../data")
DS = (DATA / "table_yolo").resolve()
GPU = torch.cuda.is_available()
EPOCHS = 30 if GPU else 8
print("GPU:", GPU, "| epochs:", EPOCHS)
CLASSES = DS.joinpath("classes.txt").read_text().split()
val_images = sorted((DS / "images" / "val").glob("*.jpg"))
truth = json.loads((DS / "ground_truth.json").read_text())

## 1. Boxes and IoU

A detector outputs boxes `(x1, y1, x2, y2)` with a class and a confidence. To compare a predicted box with a true box we use the **Intersection over Union**:
$$ \mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} \in [0, 1]. $$
A detection is usually counted as correct (*true positive*) if IoU ≥ 0.5 with a true box of the same class.

### Exercise 1.1
Implement `iou(a, b)` for two boxes `[x1, y1, x2, y2]` (no loops needed). Check it against `torchvision.ops.box_iou`. Compute **by hand first** the IoU of `[0, 0, 4, 4]` and `[2, 2, 6, 6]`.

In [ ]:
def iou(a, b):
    # TODO
    ...


print(iou([0, 0, 4, 4], [2, 2, 6, 6]))
assert abs(iou([0, 0, 4, 4], [2, 2, 6, 6]) - 4 / 28) < 1e-9
rng = np.random.default_rng(0)
for _ in range(100):
    a = np.sort(rng.uniform(0, 100, (2, 2)), axis=0).T.ravel()[[0, 2, 1, 3]]
    b = np.sort(rng.uniform(0, 100, (2, 2)), axis=0).T.ravel()[[0, 2, 1, 3]]
    ref = box_iou(torch.tensor([a]), torch.tensor([b])).item()
    assert abs(iou(a, b) - ref) < 1e-6
print("IoU OK")

## 2. Non-maximum suppression

A detector produces many overlapping boxes for the same object. **NMS** keeps the most confident box and removes the others that overlap it by more than a threshold, then repeats.

### Exercise 2.1
Implement `my_nms(boxes, scores, thr)` returning the indices kept (in decreasing score order). Compare with `torchvision.ops.nms`.

In [ ]:
def my_nms(boxes, scores, thr=0.5):
    # TODO
    ...


boxes = np.array([[10, 10, 50, 50], [12, 12, 52, 48], [100, 100, 140, 150], [11, 9, 49, 51], [98, 105, 139, 148]], float)
scores = np.array([0.90, 0.75, 0.80, 0.60, 0.95])
kept = my_nms(boxes, scores, 0.5)
ref = nms(torch.tensor(boxes), torch.tensor(scores), 0.5).tolist()
print("kept:", kept, "| torchvision:", ref)
assert kept == ref

## 3. Why fine-tune?

### 3.1 A detector pre-trained on COCO
YOLO11n was trained on COCO (80 everyday classes: person, car, cup…). What does it see on our table?

In [ ]:
coco_model = YOLO("yolo11n.pt")  # downloads 5 MB the first time
res = coco_model.predict(str(val_images[0]), conf=0.25, verbose=False)[0]
print("COCO detections:", [(coco_model.names[int(c)], round(float(p), 2)) for c, p in zip(res.boxes.cls, res.boxes.conf)])
plt.figure(figsize=(10, 5.6)); plt.imshow(res.plot()[:, :, ::-1]); plt.axis("off");

### 3.2 The hand-made detector of Day 1
The HSV + contour detector seen on Day 1 (Lecture 2), applied to the same images. Count how many parts it finds compared with the truth.

In [ ]:
def hsv_detector(img_bgr, min_area=300):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 90, 40), (179, 255, 255))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return [c for c in contours if cv2.contourArea(c) >= min_area]


found = sum(len(hsv_detector(cv2.imread(str(f)))) for f in val_images)
total = sum(len(truth[f.name]) for f in val_images)
n_washers = sum(p["class"] == "washer" for f in val_images for p in truth[f.name])
print(f"HSV detector: {found} blobs for {total} parts ({n_washers} of them are grey washers)")

**Question.** Give two reasons why the Day 1 detector fails here, and what it would cost to fix it by hand.

## 4. The dataset

Ultralytics expects `images/{train,val}` and `labels/{train,val}` with one `.txt` per image. Our labels are in the **segmentation** format: `class x1 y1 x2 y2 …` (polygon, coordinates normalised by the image width/height). For detection, Ultralytics derives the boxes from the polygons.

### Exercise 4.1
Read the label file of `val_000.jpg`, convert each polygon to a **pixel** bounding box and draw it.

In [ ]:
img = cv2.imread(str(val_images[0]))
H, W = img.shape[:2]
label_file = DS / "labels" / "val" / (val_images[0].stem + ".txt")
for line in label_file.read_text().splitlines():
    v = line.split()
    cls = int(v[0])
    # TODO
    x1, y1, x2, y2 = ...
    cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)
    cv2.putText(img, CLASSES[cls], (int(x1), int(y1) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
plt.figure(figsize=(10, 5.6)); plt.imshow(img[:, :, ::-1]); plt.axis("off");

yaml_path = Path("table_yolo.yaml")
yaml_path.write_text(f"path: {DS.as_posix()}\ntrain: images/train\nval: images/val\n"
                     + "names:\n" + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES)))
print(yaml_path.read_text())

## 5. Fine-tune YOLO11n

`YOLO("yolo11n.pt")` loads the COCO weights; `train()` replaces the classification part of the head by our 3 classes and fine-tunes everything. Ultralytics applies its own augmentations (mosaic, HSV jitter, flips, scaling).

In [ ]:
t0 = time.time()
det_model = YOLO("yolo11n.pt")
det_model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=640, batch=16, workers=0, seed=0,
                project="runs", name="detect", exist_ok=True, plots=False, verbose=False)
print(f"training time: {time.time() - t0:.0f}s")
metrics = det_model.val(data=str(yaml_path), verbose=False, plots=False)
print(f"precision {metrics.box.mp:.3f} | recall {metrics.box.mr:.3f} | "
      f"mAP50 {metrics.box.map50:.3f} | mAP50-95 {metrics.box.map:.3f}")
for c, ap in zip(CLASSES, metrics.box.maps):
    print(f"  {c:7s} mAP50-95 {ap:.3f}")

**Questions.**
1. What is the difference between mAP@0.5 and mAP@0.5:0.95? Why is the second one lower?
2. For a robot that must grasp the parts, which matters more: a high recall or a high precision?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4.6))
for a, f in zip(ax, val_images[:2]):
    r = det_model.predict(str(f), conf=0.5, verbose=False)[0]
    a.imshow(r.plot()[:, :, ::-1]); a.axis("off")

## 6. From boxes to millimetres

We reuse the calibration of Day 1 (`calibration.yaml`: $K$ and the distortion of the USB camera) and the 4 ArUco markers of the table (centres at (0, 0), (400, 0), (400, 280), (0, 280) mm).

### Exercise 6.1
Complete `pixels_to_mm(img, uv)`: detect the markers, undistort their **corners** and the points `uv`, estimate the homography and transform the points. (Same steps as Lab 4.)

**Robustness.** In 6 of the 40 validation images, marker 2 (bottom-right, at the edge of the field of view, in a dark corner) is **not detected** — this happens on real set-ups too. Instead of the 4 marker centres, use the **4 corners of every detected marker** (corner order: top-left, top-right, bottom-right, bottom-left; side 40 mm) with `cv2.findHomography`: any 1 marker already gives 4 points, and more points average the noise.

In [ ]:
fs = cv2.FileStorage(str(DS / "calibration.yaml"), cv2.FILE_STORAGE_READ)
K, dist = fs.getNode("K").mat(), fs.getNode("dist").mat()
fs.release()
MARKERS_MM = {0: (0.0, 0.0), 1: (400.0, 0.0), 2: (400.0, 280.0), 3: (0.0, 280.0)}
HALF = 20.0  # half side of a marker [mm]
CORNER_OFFSETS = np.array([[-HALF, -HALF], [HALF, -HALF], [HALF, HALF], [-HALF, HALF]])  # TL, TR, BR, BL
aruco = cv2.aruco.ArucoDetector(cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50))


def undistort_px(uv):
    return cv2.undistortPoints(np.asarray(uv, np.float32).reshape(-1, 1, 2), K, dist, P=K).reshape(-1, 2)


def pixels_to_mm(img_bgr, uv):
    corners, ids, _ = aruco.detectMarkers(img_bgr)
    # TODO
    ...


errors, missed = [], 0
for f in val_images:
    img = cv2.imread(str(f))
    r = det_model.predict(img, conf=0.5, verbose=False)[0]
    xyxy = r.boxes.xyxy.cpu().numpy()
    cls = r.boxes.cls.cpu().numpy().astype(int)
    centres = np.c_[(xyxy[:, 0] + xyxy[:, 2]) / 2, (xyxy[:, 1] + xyxy[:, 3]) / 2]
    xy = pixels_to_mm(img, centres) if len(centres) else np.zeros((0, 2))
    for p in truth[f.name]:
        same = [k for k in range(len(cls)) if CLASSES[cls[k]] == p["class"]]
        d = [np.hypot(*(xy[k] - [p["x_mm"], p["y_mm"]])) for k in same]
        if d and min(d) < 15:
            errors.append(min(d))
        else:
            missed += 1
errors = np.array(errors)
print(f"{len(errors)} parts located, {missed} missed | error: mean {errors.mean():.2f} mm, "
      f"95th percentile {np.percentile(errors, 95):.2f} mm, max {errors.max():.2f} mm")
plt.hist(errors, bins=30); plt.xlabel("position error [mm]"); plt.ylabel("parts");

**Question.** The box centre is not exactly the centre of the part on the table. Why? (Think about perspective and distortion.) How could the segmentation mask help?

## 7. Segmentation: shape and orientation for the gripper

`yolo11n-seg` predicts, for each object, a box **and a mask**. Our labels already contain polygons, so the same dataset works.

### Exercise 7.1
Train the segmentation model, then for each predicted **plate**: transform its mask outline (`r.masks.xy[k]`, pixel polygon) to millimetres with `pixels_to_mm`, and compute its centroid and orientation **on the table plane** with the image moments of Day 1 (`cv2.moments` on the polygon in mm, $\theta = \tfrac12 \operatorname{atan2}(2\mu_{11}, \mu_{20} - \mu_{02})$). Compare with the truth (angles modulo 180°).

In [ ]:
t0 = time.time()
seg_model = YOLO("yolo11n-seg.pt")
seg_model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=640, batch=16, workers=0, seed=0,
                project="runs", name="segment", exist_ok=True, plots=False, verbose=False)
print(f"training time: {time.time() - t0:.0f}s")
m = seg_model.val(data=str(yaml_path), verbose=False, plots=False)
print(f"box mAP50-95 {m.box.map:.3f} | mask mAP50-95 {m.seg.map:.3f}")

In [ ]:
def angle_diff(a, b):
    d = (a - b) % 180
    return min(d, 180 - d)


pos_err, ang_err = [], []
for f in val_images:
    img = cv2.imread(str(f))
    r = seg_model.predict(img, conf=0.5, verbose=False)[0]
    if r.masks is None:
        continue
    for k, c in enumerate(r.boxes.cls.cpu().numpy().astype(int)):
        if CLASSES[c] != "plate":
            continue
        # TODO
        xc, yc, theta = ...  (mm, mm, degrees) on the table plane
        plates = [p for p in truth[f.name] if p["class"] == "plate"]
        best = min(plates, key=lambda p: np.hypot(p["x_mm"] - xc, p["y_mm"] - yc))
        pos_err.append(np.hypot(best["x_mm"] - xc, best["y_mm"] - yc))
        ang_err.append(angle_diff(theta, best["angle_deg"]))
print(f"{len(pos_err)} plates | centroid error {np.mean(pos_err):.2f} mm | orientation error "
      f"mean {np.mean(ang_err):.1f} deg, max {np.max(ang_err):.1f} deg")

In [ ]:
r = seg_model.predict(str(val_images[1]), conf=0.5, verbose=False)[0]
plt.figure(figsize=(10, 5.6)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off");

## 8. Deployment

`det_model.export(format="onnx")` produces a file usable from C++, OpenCV DNN, ONNX Runtime, TensorRT… The path of the best weights is given by `det_model.trainer.best` (somewhere under `runs/`): **download them from Colab** for your project. On the robot PC, the loop is: grab frame → `model.predict` → `pixels_to_mm` → send `(X, Y, θ)` to the robot.